# 第12回 演習：論文のコードを読んで、動かす


---


## 前回からの接続


第9回から第11回まで、PCAで縮め、地図に置き、クラスタリング（clustering）でグループに分けてきました。どれも、自分で書いたコードでした。

今日は、研究者が論文といっしょに公開したコードを読みます。題材は、ハッブル宇宙望遠鏡が撮った銀河の画像です。「この論文は何を主張していて、そのコードのどこがそれを確かめているのか」を、コードをブロックに分けて読みながら追いかけます。


---


## 今日の分析目標


あなたは天文学の研究室に入ったばかりです。先輩から「この論文のコードを動かして、主張が本当か確かめておいて」と頼まれました。

論文（Uzeirbegovic, Geach, Kaviraj 2020, MNRAS）は、「銀河の形は、十数枚の固有銀河（銀河の画像の主成分）の足し算で表せる」と主張しています。著者のノートを取ってきて、ブロックごとに読み、動かし、論文の図を自分の手元で描きます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは天文学の研究室に入ったばかりの新人です。先輩から、銀河の形を調べた論文を1本渡され、「この論文のコードを動かして、主張が本当か確かめておいて」と頼まれました。

決めたいのは、論文の主張が手元で動かしても成り立つかどうかです。あわせて、著者のコードのどこがその主張を確かめているのかも、知っておきたいところです。


### 手元のデータ

著者がGitHubに公開したリポジトリ（https://github.com/emiruz/eigengalaxies）から、著者のノートと、ハッブル宇宙望遠鏡が撮った銀河の画像1,222個ぶんを取ってきます。1個の銀河が、縦30×横30画素で3つの波長の帯をもつ画像で、解析では1個を1行にした表にします。答えの列はありません。


### 最後に出す答え

固有銀河いくつで銀河の形の何%を表せるかを、論文の数と比べます。あわせて、似た形の銀河を集めると、どんなグループに分かれるかを示します。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 論文に付いたノートを、見出しと関数のまとまり（ブロック）に分けて、全体の地図を描ける
- `.ipynb`がJSONという形式のファイルで、中身を辞書とリストとして読めることを説明できる
- `glob`でたくさんのファイルを集め、`.npy`とPNGを、形式に合った方法で読める
- 前処理（明るさのクリップ、回転をそろえる）→ PCA → 外れ値（outlier） → クラスタリング、というパイプライン（pipeline）の各段が何をしているか言える
- 他人のコードを再現するときに、ファイルの並び順や乱数（random number）のように、結果が揺れる所に気づける


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import os
import json
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 論文のコードとデータを、取ってくる

- 役割：著者が公開したリポジトリから、ノートと、銀河1,222個ぶんの画像だけを取ってくる
- 入力：リポジトリの場所（URL）と、取ってくるファイルの名前の型
- 出力：`eigengalaxies`というフォルダ（そこを作業の場所にする）

リポジトリ全体は400MB以上あるので、`git`の機能で必要なファイルだけを取ってきます（数秒・約40MB）。`!`で始まる行は、Pythonではなく、計算機に直接出す命令です。`os.chdir`は、作業の場所（フォルダ）を移す関数です。著者のコードはこのフォルダの中で動く前提なので、移しておきます。


In [ ]:
WORK = '/content'   # Colab の作業場所
URL = 'https://github.com/emiruz/eigengalaxies.git'
PICK = ("'/Eigengalaxies.ipynb' '/CANDELS/GDS_[12]???.npy' "
        "'/GDS_RGBs/GDS_0[12]???.png'")      # 取ってくるファイルの型
os.chdir(WORK)
if not os.path.exists('eigengalaxies'):     # まだ取ってきていなければ
    !git clone -q --depth 1 --filter=blob:none --sparse $URL
    !cd eigengalaxies && git sparse-checkout set --no-cone {PICK}
os.chdir('eigengalaxies')                   # ここを作業の場所にする
print(sorted(os.listdir('.')))


`CANDELS`（銀河の画像の数値）、`GDS_RGBs`（カラーの縮小画像）、`Eigengalaxies.ipynb`（著者のノート）がそろいました。


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、他人の解析の①〜③を、コードから読み解きます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = True
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. 論文とコードの地図


3つのファイルがそろいました。では、著者のノートのどこから読めばよいでしょうか。他人のコードは、いきなり1行目から読みません。まず、どんなまとまり（ブロック）がどの順に並んでいるかをつかみます。

じつは、ノート（`.ipynb`）はJSONという形式の文字のファイルです。JSONは、辞書（`{名前: 値}`）とリスト（`[…]`）を入れ子にして、データを書く形式です。`json.load`で読むと、Pythonの辞書とリストになります。


### お手本：著者のノートを、JSONとして読む

- 役割：著者のノートを開き、中身がどんな形で入っているかを見る
- 入力：`Eigengalaxies.ipynb`
- 出力：辞書のキーと、セルの数


In [ ]:
nb = json.load(open('Eigengalaxies.ipynb'))   # ノートの中身は JSON
print(list(nb.keys()))
print(len(nb['cells']), 'セル')
print(nb['cells'][2])                         # 2番目のセルの中身


辞書のキーに`'cells'`があり、そこにセルが35個、リストで入っています。1つのセルも辞書で、`'cell_type'`（`'markdown'`か`'code'`）と、`'source'`（中身の文字を、行ごとに分けたリスト）を持っています。


### 変えてみる：見出しだけを並べて、地図にする

`for`と`if`で、文章のセル（markdown）だけを取り出し、先頭の40字を表示します。`''.join(…)`は、行ごとに分かれた文字のリストを、1つの文字につなげる書き方です。`.strip()`は、前後の空白や改行を取り除きます。


In [ ]:
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'markdown':
        print(i, ''.join(cell['source']).strip()[:40])


見出しを並べると、ノートの地図が見えます。

| セル | 見出し | 今日 |
|---|---|---|
| 1・3 | Common imports／Constants（道具と定数） | 使う |
| 5 | Data Processing Functions（前処理の関数） | 使う（2・3節） |
| 7 | Image Generation（画像を並べる関数） | 使う |
| 9・11 | Data／PCA | 使う（3・4節） |
| 14・18 | Checks／Sample size（PCAを何百回もくり返す確認） | 重いので飛ばす |
| 16 | Outlier Detection（外れ値） | 使う（5節） |
| 20・24 | 累積寄与率（cumulative proportion of variance explained）／The Eigengalaxies | 使う（4節） |
| 26 | Imputing Values（欠けた値の補い） | 別の道具が要るので飛ばす |
| 33・34 | Affinity Propagation Clustering | 使う（6節） |

セルの番号は、0から数えています。


### 著者のセルを、読んでから実行する

著者のコードを写して打ち直すと、写し間違えます。そこで、ノートのJSONからセルの中身を取り出して、そのまま実行します。

- `author(i)`は、i番目のセルの中身を1つの文字にして返す関数です
- `print(author(i))`で読み、`exec(author(i))`で実行します。`exec`は、文字で書かれたコードを、その場で実行する関数です


In [ ]:
def author(i):                       # i 番目のセルの中身
    return ''.join(nb['cells'][i]['source'])

print(author(3))                      # 定数のセルを読む
exec(author(1))                       # 道具を呼び出すセルを実行
exec(author(3))                       # 定数のセルを実行
print(SHP, BANDS)


`SHP=(30,30,3)`は、1つの銀河の画像の大きさ（縦30×横30画素×3つの色の帯）です。`BANDS`は、ハッブル望遠鏡が撮った3つの波長の帯の名前です。


### TODO①：コードのセルの、1行目を並べる

お手本をまねて、今度はコードのセル（`'code'`）だけを取り出し、番号と1行目を表示してください。どのセルが何をしているか、1行目からどこまで分かるでしょうか。


In [ ]:
# TODO: コードのセルの番号と、1行目を表示する
# ヒント: if cell['cell_type'] == 'code': の中で
#         cell['source'][0] が1行目（改行は .strip() で消す）
...


コードのセルは19個あります。`def rescale(a):`（5）、`import matplotlib.image as mpimg`（7）、`raw_data,indexes=get_data()`（9）、`pca=PCA(.96).fit(norm_data)`（11）のように、1行目だけでも「関数を作るブロック」と「それを使って処理するブロック」が見分けられます。


---


## 2. 読み込み：形式によって、読み方が違う


ノートの地図ができました。では、データはどうやって読み込まれているのでしょうか。データは、2つの形式で入っています。

- `CANDELS/GDS_1000.npy`など：numpyの配列を、そのまま保存した形式（.npy）。`np.load`で読む
- `GDS_RGBs/GDS_01000.png`など：ふつうの画像の形式（PNG）。`plt.imread`で読む


### お手本：1つの銀河を、2つの形式で読む

- 役割：同じ銀河の .npyとPNGを読み、大きさと値の範囲を比べる
- 入力：2つのファイル名
- 出力：配列の形・型・最小と最大


In [ ]:
x = np.load('CANDELS/GDS_1000.npy', allow_pickle=True)
im = plt.imread('GDS_RGBs/GDS_01000.png')
print(x.shape, x.dtype, x.min().round(3), x.max().round(3))
print(im.shape, im.dtype, im.min(), im.max())


どちらも30×30×3の配列ですが、中身が違います。.npyは望遠鏡が測った明るさの数値そのもの（−0.015〜0.185、マイナスもある）で、PNGは人が見るために0〜1に直したカラー画像です。解析には .npy、目で見るにはPNGを使います。


### 変えてみる：たくさんのファイルを、globで集める

`glob('CANDELS/*.npy')`は、`*`の所に何が入ってもよい、という型に合うファイルの名前を、すべてリストにして返します。


In [ ]:
files = glob('CANDELS/*.npy')
print(len(files))
print(files[:3])


1,222個。ただし、並びは`GDS_1000`、`GDS_1001`… の順とは限りません。`glob`が返す順番は、計算機やファイルの置かれ方によって変わることがあります。覚えておいてください。


### お手本：著者の読み込みの関数を読む

5番のセル（Data Processing Functions）には、関数が5つ入っています。今日の読み込みに使うのは`get_data`です。まず読んで、ブロックに分けます。


In [ ]:
print(author(5)[author(5).index('def get_data'):
                  author(5).index('def get_catalog')])


`get_data`を、ブロックに分けて読みます。

1. 名前を集める：`glob`で .npyのファイルを集め、`re.search`で`GDS_1000`のような名前の部分を取り出す（`re`は、文字の型で検索する道具）
2. 1つずつ読む：`for`で、`np.load`してから`flatten()`で2,700個（30×30×3）の横一列にし、リスト`L`にためる。3つの帯のどれかが真っ暗（最大が0）なら、`skip`にTrueを記録する
3. 表にして、外す：`np.row_stack(L)`で縦に積んで表にし、明るさが一様（uniform）（標準偏差が0）なものと`skip`のものを外す
4. 返す：表と、名前の並び


### 著者の関数で読み込み、並べ方をそろえる

5番のセルを実行して関数を作り、`get_data`を呼びます。そのあと、こちらで1段足して、名前の順に並べ直します。`glob`の順番が計算機で変わると、このあとの結果の並びも変わってしまうからです。`np.argsort`は、並べ替えたときの順番（番号）を返します。


In [ ]:
exec(author(5))                        # 前処理の関数を作る
raw_data, indexes = get_data()
order = np.argsort(indexes)            # 名前の順に並べる順番
raw_data, indexes = raw_data[order], indexes[order]
print(raw_data.shape, indexes[:3])


1,222行 × 2,700列の表になりました。1行が1つの銀河です。この1,222個では、外された銀河はありませんでした。


### おさらい：[ ] の中のfor（内包表記）

```python
out = [i * 10 for i in range(3)]   # out は [0, 10, 20]
```

`[式 for 名前 in 並び]`と書くと、並びの中身を1つずつ名前に入れて式を計算し、その結果をリストに集めます（内包表記、第5回）。`out = []`から`for`の中で`append`していくのと同じことを、1行で書けます。


### TODO②：外す条件を、1つの銀河で確かめる

`get_data`が外す2つの条件を、お手本で読んだ`x`（`GDS_1000`）で確かめてください。① 3つの帯それぞれの最大が0か（`x[:,:,i].max() == 0`をi＝0, 1, 2で）、② 全体の標準偏差（standard deviation）が0でないか。


In [ ]:
# TODO: x の3つの帯の最大が0かどうかと、標準偏差が0でないかを表示する
# ヒント: [x[:, :, i].max() == 0 for i in range(3)]、x.std() != 0
...


結果は`[False, False, False]`と`True`。どの帯も真っ暗ではなく、明るさにばらつきもあるので、この銀河は外されません。著者は、撮影の失敗で一部の帯が写っていない画像を、こうして機械的に除いていたのです。


---


## 3. 前処理：明るさをそろえ、向きをそろえる


1,222個の銀河を、表に読み込めました。では、PCAの前に何をしているのでしょうか。9番のセル（Data）は、`normalise_all(raw_data, 0, .99)`を呼んでいます。この関数は、2つの前処理をします。

- 明るさのクリップ：全体の明るさの0%点〜99%点の外を、その値に切りそろえる（`np.clip`）。ごく一部の、とても明るい画素に引っぱられないため
- 回転をそろえる：銀河ごとに、長く伸びている向きを求め、同じ向きに回す（`rotation`）。同じ形なのに向きだけ違う銀河を、「違う形」と数えないため


### お手本：著者の前処理を実行し、前と後を並べる

- 役割：画像を並べる関数（7番のセル）を作り、前処理の前と後の9個を、並べて見る
- 入力：`raw_data`
- 出力：前処理した表`norm_data`と、2枚の画像の並び

`make_image_pre(表, (3, 3))`は、表の先頭9行を、3×3の画像に並べて描く関数です（7番のセル）。

ここで`.copy()`（写し）を渡しているのには、わけがあります。7番のセルを読むと、`make_image_pre`は描く前に`r-=r.min()`のように、受け取った配列の中身をその場で書きかえています。`raw_data[:9]`のように表の一部をそのまま渡すと、元の表の値まで書きかわり、あとのPCAの結果が変わってしまいます（実際に試すと、主成分の数が10個から12個に変わります）。他人の関数を使うときは、渡したものを書きかえないかを読んで確かめます。


In [ ]:
exec(author(7))                          # 画像を並べる関数を作る
norm_data = normalise_all(raw_data, 0, .99)
make_image_pre(raw_data[:9].copy(), (3, 3), '前')   # 写しを渡す
make_image_pre(norm_data[:9].copy(), (3, 3), '後')
plt.show()


前処理の後では、伸びた銀河の多くが、縦の向きに回されています（丸い銀河は、向きがはっきりしないので、見た目はほとんど変わりません）。ただし、9個のうち3個は、回転の後も縦になっていません。明るさも、一部のとても明るい点に引っぱられにくくなりました。


### 回転の角度は、PCAで求めている

`get_angle`の中身を読むと、ここにもPCAが出てきます。

```python
def get_angle(d):
    fit=PCA(n_components=2).fit(
        np.argwhere(d>=np.quantile(d,.75)))
    return np.arctan2(*fit.components_[0])
```

- `np.argwhere(d >= np.quantile(d, .75))`：明るさが上位25%の画素の座標（縦・横の番号）を集める
- その座標の点の集まりにPCAをかける。第1主成分（principal component）は、点がいちばん長く散らばる向き、つまり銀河が伸びている向き
- `np.arctan2`で、その向きを角度に直す

PCAは「データがいちばん散らばる向き」を探す道具でした（第9回）。ここでは画素の座標に使って、銀河の向きを測っています。


### 変えてみる：1つの銀河の角度を測る

はっきり細長い銀河`GDS_1601`を読み、角度を測ります。`rotation`と同じく、画素ごとに3つの帯のいちばん明るい値を取った画像`fl`を作ってから、`get_angle`を呼びます。`np.degrees`は、角度の単位を度に直します。


In [ ]:
gal = np.load('CANDELS/GDS_1601.npy', allow_pickle=True)  # 細長い銀河
fl = np.apply_along_axis(max, 2, gal)
print(round(np.degrees(get_angle(fl)), 1))


`GDS_1601`は、46.3度の向きに伸びていました。`rotation`は、この角度に合わせて回し、縦（90度）の向きにそろえます。回転の後に測ると、90.5度です。


### 変えてみる：全部の銀河で、回転の前と後の角度を比べる

2節でおさらいした`[ ]`の中の`for`で、1行（1つの銀河）ずつ角度を測ってリストに集め、`np.median`（真ん中の値）で比べます。角度は向きなので、絶対値（`np.abs`）で見ます。


In [ ]:
def angle(row):                      # 1つの銀河の、伸びた向き（度）
    fl = np.apply_along_axis(max, 2, row.reshape(SHP))
    return np.degrees(get_angle(fl))

before = np.abs([angle(r) for r in raw_data])
after = np.abs([angle(r) for r in norm_data])
print(np.median(before).round(1), np.median(after).round(1))


回転の前は、角度の真ん中が40度台で、向きはばらばらです。回転の後は、真ん中が90度近く（縦）に集まります。回転の後は全体の約7割が90度±10度に入り、回転の前は1割未満でした。著者の前処理は、ねらいどおり向きをそろえていました。


In [ ]:
# @title 図：明るい画素の座標と、第1主成分の向き
from sklearn.decomposition import PCA as _PCA
pts = np.argwhere(fl >= np.quantile(fl, .75))
p = _PCA(n_components=2).fit(pts)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.imshow(fl, cmap='gray')
ax.scatter(pts[:, 1], pts[:, 0], s=6, color='#e63946', label='明るい画素')
c = pts.mean(axis=0); v = p.components_[0] * 10
ax.annotate('', xy=(c[1] + v[1], c[0] + v[0]), xytext=(c[1] - v[1], c[0] - v[0]),
            arrowprops=dict(arrowstyle='<->', color='#0066cc', lw=3))
ax.set_title('第1主成分＝伸びている向き'); ax.legend(fontsize=10)
plt.show()


---


## 4. PCAと固有銀河


前処理した1,222個 × 2,700列の表に、PCAをかけます。第9回の顔と同じく、主成分は2,700個の数の並びなので、30×30×3に戻せば画像として見られます。これが固有銀河です。


### お手本：著者のPCAのセルを読んで、実行する

- 役割：寄与率（proportion of variance explained）の合計が96%になるまでの主成分を求め、各銀河の得点を出す
- 入力：`norm_data`
- 出力：`pca`・`scores`（得点）・`lik`（あとの5節で使う）と、主成分の数

`PCA(.96)`のように、`n_components`に0〜1の数を渡すと、「累積寄与率がこの割合に届くまでの数」を自動で選びます。


In [ ]:
print(author(11))
exec(author(11))


10個の固有銀河で、1,222個の銀河の散らばりの96%を表せました。論文は約1万個の銀河で約12個としており、近い値です。


### 変えてみる：固有銀河を描く（24番のセル）

24番のセルは、10個の主成分を、3つの帯ごとに画像にして縦に並べます。


In [ ]:
exec(author(24))


1行目（第1主成分）は、大きく広がった明るさです。銀河全体の明るさと大きさにあたります。2行目は、縦に伸びた明るい中心で、回転を縦にそろえたので、「縦に伸びているかどうか」が2番目に大きな違いとして現れました。3行目は、明るさが中心にどれだけ集まっているか、5行目あたりは縦と横の明るさの対比、それより後ろは、上下の片寄りのような細かい形の違いです。


### TODO③：90%を表すには、何個か

`PCA()`（すべての主成分）を`norm_data`にかけて累積寄与率を求め、初めて0.9以上になる主成分の数を求めてください（第9回と同じ書き方です）。


In [ ]:
# TODO: 累積寄与率が初めて 0.9 以上になる主成分の数を求める
# ヒント: p = PCA().fit(norm_data)
#         cum = np.cumsum(p.explained_variance_ratio_)
#         np.argmax(cum >= 0.9) + 1
...


3個です。第1主成分だけで76.3%、3個で90%を超えます。96%に10個、99%には45個が要ります。銀河の形のほとんどはごく少ない固有銀河で表せるという、論文の主張どおりです。


### お手本：組み立て直して、確かめる

固有銀河の足し算で、本当に元の形に戻るかを見ます。`inverse_transform`は、得点から元の2,700列の形に戻すメソッドです（第9回の再構成と同じ計算）。


In [ ]:
for k in [2, 10, 50]:
    p = PCA(k).fit(norm_data)
    rec = p.inverse_transform(p.transform(norm_data[:9]))
    make_image_pre(rec, (3, 3), f'{k}個')
plt.show()


2個ではぼんやりした楕円ですが、10個で、銀河ごとの伸び方や明るさの偏りまで戻ります。50個では、前処理の後の画像とほとんど同じです。


---


## 5. 外れ値：めずらしい銀河を探す


10個の固有銀河で、銀河の形がほぼ戻りました。では、うまく戻らない銀河はあるでしょうか。`pca.score_samples`は、各銀河がこのPCAのつくる「ふつうの散らばり」にどれくらいありそうかを、数にします（大きいほどありそう）。いちばん小さい銀河は、固有銀河の足し算ではうまく表せない、めずらしい形の銀河です。


### お手本：著者の外れ値のセルを実行する

- 役割：ありそうさ`lik`の小さい順・大きい順に、25個ずつカラー画像を並べる
- 入力：`lik`・`indexes`
- 出力：2枚の画像の並び（めずらしい25個、ありふれた25個）

16番のセルの`make_image`は、名前の並びに合うPNGを開いて並べる関数です（7番のセル）。


In [ ]:
print(author(16))
exec(author(16))
plt.show()


1枚目（めずらしい25個）には、とても明るく青い点（星が写りこんだのかもしれません）、2つ以上が1枚に写ったもの、大きく広がった明るい銀河が並びます。2枚目（ありふれた25個）は、小さく暗い銀河ばかりで、背景のざらつきが目立ちます。外れ値を探すと、めずらしい銀河だけでなく、データの汚れ（写りこみ・重なり）も見つかります。


---


## 6. クラスタリング：似た形の銀河を集める


めずらしい銀河は見ました。では、似た形の銀河どうしは、どう集まるでしょうか。最後に、固有銀河の得点`scores`で、似た形の銀河をグループに分けます。著者はAffinity Propagationという方法を使っています。k-meansと違って、グループの数を先に決めません。各点が「自分の代表にふさわしい点」を互いに推しあい、代表と、そこに集まるグループが決まります。


### お手本：著者のクラスタリングのセルを実行する

- 役割：Affinity Propagationで、得点からグループを求め、大きいグループを数える
- 入力：`scores`
- 出力：`clust`（グループの番号は`clust.labels_`）と、大きいグループ5つの大きさ

`value_counts()`は、値ごとの数を、多い順に並べるメソッドです。


In [ ]:
print(author(33))
exec(author(33))
sizes = pd.Series(clust.labels_).value_counts()
print(len(sizes), 'グループ')
print(sizes.head(5).values)


約145のグループができました。いちばん大きいグループは143個、次が141個、110個、97個…と続き、1個だけのグループもたくさんあります。

グループの数は、実行するたびに少し揺れることがあります（145〜147ほど）。著者のセルは、乱数の種（`random_state`）を決めていないからです。大きいグループの顔ぶれは、ほぼ変わりません。


### 変えてみる：いちばん大きいグループを、画像で見る

グループの番号`sizes.index[0]`の銀河の名前を取り出し、`make_image`で先頭の30個を並べます。


In [ ]:
top = indexes[clust.labels_ == sizes.index[0]]
make_image(top[:30], (5, 6))
plt.show()


いちばん大きいグループは、小さく丸い、中心の明るい銀河です。PCAの得点が近い銀河は、目で見ても形が似ています。


### TODO④：2番目に大きいグループを見る

お手本をまねて、2番目に大きいグループ（`sizes.index[1]`）の先頭30個を並べてください。いちばん大きいグループと、形はどう違いますか。


In [ ]:
# TODO: 2番目に大きいグループの先頭30個を、画像で並べる
# ヒント: お手本の sizes.index[0] を sizes.index[1] に変える
...


2番目のグループは、細長く伸びた銀河が中心です（並べているカラーのPNGは回転する前の画像なので、伸びる向きはばらばらです）。回転をそろえてからPCAをかけたので、「伸びているかどうか」が、そのままグループの違いになって現れました。

先輩への答えは、次のとおりです。

> 論文の主張は、手元の1,222個でも確かめられた。前処理（明るさのクリップと回転）のあと、10個の固有銀河で形の散らばりの96%を表せ（論文は約12個）、その得点で、丸い銀河・細長い銀河などのグループに分けられる。ただし、ファイルの並び順と乱数の種で結果が揺れる所があるので、再現するときは並べ方をそろえ、揺れを確かめる。

まだ言えないことは、次の2つです。

- 前処理の効果：回転をそろえる段がなかったら、固有銀河の数はどう変わるか（課題）
- グループの数は妥当か：145は多すぎないか。分け方の良し悪しを数で測るには


---


## 目標に答えられたか


- 今日の目標は「論文のコードを動かして、主張が本当か確かめる」ことでした。4節で、何個の固有銀河で何割を表せましたか？ 論文の値と比べてどうでしたか？
- 1節で、著者のノートを、どうやってブロックに分けましたか？`.ipynb`の中身はどんな形でしたか？
- 2節で、.npyとPNGは、何が違いましたか？`glob`の順番について、何に気をつけましたか？
- 3節で、回転の角度は、何にPCAをかけて求めていましたか？


---


## 今日の要点


- 他人のコードは、まず見出しと関数のまとまりで地図を描いてから、ブロックごとに読む
- `.ipynb`はJSON。辞書とリストの入れ子として読める
- 形式ごとに読み方が違う：.npyは`np.load`、PNGは`plt.imread`。たくさんのファイルは`glob`で集める
- パイプライン：読み込み → 明るさのクリップ → 回転をそろえる（画素の座標にPCA）→ PCA（固有銀河）→ 外れ値 → クラスタリング
- 再現するときは、ファイルの並び順や乱数の種のように、結果が揺れる所に気をつける
- Python：`json.load`、`exec`、`glob`、`np.load`、`plt.imread`、`os.chdir`


---


## 次回へ


今日のクラスタリングは、約145ものグループに分かれました。第11回の国のグループ分けでも、K＝3はこちらで決めた数でした。

次回は、分け方の良し悪しを数で測ります。その分け方は妥当か、グループの数をいくつにすべきか、根拠をもって言えるでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

著者の前処理`normalise_all`は、「明るさのクリップ」と「回転をそろえる」の2段の処理をしていました。今度は回転をそろえる段を外し、明るさのクリップだけをした表`clip_only`を作って、`PCA(.96)`をかけてください。主成分の数と、上位3つの寄与率を表示し、固有銀河を描いてください（24番のセルは`pca`を描くので、`pca`を差し替えてから`exec(author(24))`します）。

`np.quantile(raw_data, [0, .99])`で、クリップの下と上の値が求まります（`normalise_all`の中と同じ）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`L, U = np.quantile(raw_data, [0, .99])` → `clip_only = np.clip(raw_data, L, U)`。
`pca_r = PCA(.96).fit(clip_only)`で主成分の数は`pca_r.n_components_`。
描くときは、4節の`pca`を取っておいて（`pca_keep = pca`）、`pca = pca_r`にしてから`exec(author(24))`。描き終えたら`pca = pca_keep`に戻します。

</details>


### 応用②（判断）

応用①の、回転をそろえないときの主成分の数（寄与率96%まで）はいくつですか。整数で答えてください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`pca_r.n_components_`を表示します。4節（回転をそろえたとき）は10個でした。

</details>


### 応用③（解釈）

回転をそろえたとき（4節）と、そろえないとき（応用①）で、必要な固有銀河の数と、固有銀河の形はどう変わりましたか。「回転をそろえる」という前処理の1段が、この論文の主張にとってなぜ要るのかを、研究室の先輩に向けて3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### 脳の画像を、区画に分ける（Abrahamほか2014）

医用画像にも、同じ「読み込み → 前処理 → 次元の扱い → クラスタリング」のパイプラインがあります。Abrahamほか（2014, Frontiers in Neuroinformatics「Machine learning for neuroimaging with scikit-learn」）は、安静にしている人の脳のfMRI（脳の活動の時間変化を撮るMRI）を、似た動きをする場所ごとの区画に分けました。

論文のコードは古い版の道具で書かれていて今は動かないので、同じ手順を今の道具（nilearnとscikit-learn）で書き直した版で、論文の最後の図（脳の区画分け）を再現します。

- 形式：脳の画像はNIfTI（`.nii.gz`）という医用画像の形式。縦×横×高さ×時間の4次元の配列です
- データ：nilearnが公開データから1人分（約44MB、20秒ほど）を取ってきます


In [ ]:
!pip install -q nilearn
from nilearn import datasets, plotting
from nilearn.maskers import NiftiMasker
import nibabel as nib

adhd = datasets.fetch_adhd(n_subjects=1)        # 1人分を取ってくる
img = nib.load(adhd.func[0])                    # NIfTI を読む
print(img.shape)                                # 縦・横・高さ・時間


61×73×61の立体（1つの画素は3mm角）を、2秒ごとに176回撮っています。


In [ ]:
# 脳の中だけを取り出し（マスク）、少しぼかし、場所ごとに標準化する
masker = NiftiMasker(mask_strategy='epi', smoothing_fwhm=6,
                     standardize='zscore_sample')
X = masker.fit_transform(img)                   # 時間 × 脳の場所
print(X.shape)


176時点 × 62,546か所の表になりました。1列が、脳の1か所（ボクセル）の、時間の動きです。


In [ ]:
from sklearn.feature_extraction.image import grid_to_graph
from sklearn.cluster import FeatureAgglomeration

mask = masker.mask_img_.get_fdata().astype(bool)
conn = grid_to_graph(*mask.shape, mask=mask)    # となり合う場所だけをつなぐ
ward = FeatureAgglomeration(n_clusters=100, connectivity=conn,
                            linkage='ward').fit(X)
labels = masker.inverse_transform(ward.labels_ + 1)   # 脳の形に戻す
plotting.plot_roi(labels, title='Ward 法で100の区画',
                  display_mode='xz', cut_coords=(0, 0))
plotting.show()


読み方：色の1つ1つが、時間の動きが似ている場所をまとめた区画です。Ward法に「となり合う場所だけをつなぐ」という条件（`connectivity`）を付けたので、どの区画もひとつながりの塊になっています。

列（場所）をグループに分けているので、使っているのは`FeatureAgglomeration`（列をまとめる版の階層クラスタリング）です。第11回の課題の階層クラスタリングと、つなぎ方（ウォード法）は同じです。

試すなら、`n_clusters`を20や500に変えて、区画の細かさがどう変わるか見てみましょう。また、つながりの条件なしのk-means（`MiniBatchKMeans(n_clusters=100).fit(X.T)`）で分けると、区画がばらばらに散らばることも確かめられます。
